In [251]:
import pandas as pd

path = "../../output/rsd/rolling_stock_final.xlsx" 
xls = pd.ExcelFile(path)
print("Available sheets:", xls.sheet_names)


Available sheets: ['train_startup_test', 'tyre_pressure', 'tyre_wear', 'airbag_pressure', 'cceb', 'air_standup', 'water_ponding', 'cardan_shaft', 'greasing_cardan_shaft']


In [252]:
selected_sheets = xls.sheet_names

dfs = {s: xls.parse(s) for s in selected_sheets}

for name, df in dfs.items():
    if name == "notification" or name == "work_order":
        continue
    print(f"{name}: shape={df.shape}")
    # print(df.head())

train_startup_test: shape=(1198, 198)
tyre_pressure: shape=(1225, 128)
tyre_wear: shape=(1198, 176)
airbag_pressure: shape=(1206, 48)
cceb: shape=(1224, 24)
air_standup: shape=(18, 11)
water_ponding: shape=(1201, 26)
cardan_shaft: shape=(1206, 20)
greasing_cardan_shaft: shape=(75, 44)


In [253]:
errors = []
bool_errors = []

def to_text(value, workorder_no, filename, column):
    try:
        if pd.isna(value):
            return ""   # store empty string instead of skipping
        return str(value)
    except Exception as e:
        errors.append({
            "workorder_no": workorder_no,
            "filename": filename,
            "column": column,
            "value": repr(value),
            "error": str(e)
        })
        # 🔥 fallback value so data is NOT lost
        return f"<<ERROR: {repr(value)}>>"



def to_bool(value, workorder_no=None, filename=None, column=None):
    try:
        if pd.isna(value):
            return False  # store False instead of skipping

        if isinstance(value, bool):
            return value

        if isinstance(value, (int, float)):
            return value == 1

        if isinstance(value, str):
            v = value.strip().lower()
            if v in ["yes", "y", "true", "1", "checked", "ok", "✓"]:
                return True
            if v in ["no", "n", "false", "0", "unchecked", "x", "✗"]:
                return False

        # unknown string or type → log and store as False
        if workorder_no and filename and column:
            bool_errors.append({
                "workorder_no": workorder_no,
                "filename": filename,
                "column": column,
                "value": value,
                "error": "Unknown boolean format, storing as False"
            })
        return False

    except Exception as e:
        # store fallback False and log error
        if workorder_no and filename and column:
            bool_errors.append({
                "workorder_no": workorder_no,
                "filename": filename,
                "column": column,
                "value": value,
                "error": str(e)
            })
        return False

### Tyre Pressure

In [254]:
df_tyrepressure = dfs.get("tyre_pressure")
tyre_pressure_json = {}

excluded_cols = ["functional_location", "work_request", "target_date"]
df_tyrepressure = df_tyrepressure[[col for col in df_tyrepressure.columns if col not in excluded_cols]]

for _, row in df_tyrepressure.iterrows():
    workorder_no = row["workorder_id"]
    filename = row["filename"]

    data = {}

    for col in df_tyrepressure.columns:
        if col in ["workorder_id", "filename"]:
            continue

        value = row[col]
        if pd.isna(value):
            continue
        
        if col.endswith("bogie_sn"):
            try:
                value = int(value)
            except (ValueError, TypeError):
                continue

        text_value = to_text(value, workorder_no, filename, col)
        
        data[col] = text_value

    if workorder_no not in tyre_pressure_json:
        tyre_pressure_json[workorder_no] = {
            "filename": filename,
            "data": data
        }
    else:
        tyre_pressure_json[workorder_no]["data"].update(data)

        if not tyre_pressure_json[workorder_no].get("filename"):
            tyre_pressure_json[workorder_no]["filename"] = filename


In [255]:
errors

[]

### Tyre Wear

In [256]:
df_tyrewear = dfs.get("tyre_wear")
tyre_wear_json = {}

excluded_cols = ["functional_location", "work_request", "target_date"]
df_tyrewear = df_tyrewear[[col for col in df_tyrewear.columns if col not in excluded_cols]]

for _, row in df_tyrewear.iterrows():
    workorder_no = row["workorder_id"]
    filename = row["filename"]

    data = {}

    for col in df_tyrewear.columns:
        if col in ["workorder_id", "filename"]:
            continue

        value = row[col]
        if pd.isna(value):
            continue
        
        if col.endswith("bogie_sn"):
            try:
                value = int(value)
            except (ValueError, TypeError):
                continue

        text_value = to_text(value, workorder_no, filename, col)

        data[col] = text_value

    if workorder_no not in tyre_wear_json:
        tyre_wear_json[workorder_no] = {
            "filename": filename,
            "data": data
        }
    else:
        tyre_wear_json[workorder_no]["data"].update(data)

        if not tyre_wear_json[workorder_no].get("filename"):
            tyre_wear_json[workorder_no]["filename"] = filename


In [257]:
errors

[]

### Airbag Pressure

In [258]:
df_airbagpressure = dfs.get("airbag_pressure")
airbag_pressure_json = {}

excluded_cols = ["functional_location", "work_request", "target_date"]
df_airbagpressure = df_airbagpressure[[col for col in df_airbagpressure.columns if col not in excluded_cols]]

errors = []

for _, row in df_airbagpressure.iterrows():
    workorder_no = row["workorder_id"]
    filename = row["filename"]

    data = {}

    for col in df_airbagpressure.columns:
        if col in ["workorder_id", "filename"]:
            continue

        value = row[col]
        if pd.isna(value):
            continue
        
        if col.endswith("bogie_sn"):
            try:
                value = int(value)
            except (ValueError, TypeError):
                continue

        text_value = to_text(value, workorder_no, filename, col)

        data[col] = text_value

    if workorder_no not in airbag_pressure_json:
        airbag_pressure_json[workorder_no] = {
            "filename": filename,
            "data": data
        }
    else:
        airbag_pressure_json[workorder_no]["data"].update(data)

        if not airbag_pressure_json[workorder_no].get("filename"):
            airbag_pressure_json[workorder_no]["filename"] = filename


In [259]:
errors

[]

### CCEB

In [260]:
df_cceb = dfs.get("cceb")
cceb_json = {}

excluded_cols = ["functional_location", "work_request", "target_date"]
df_cceb = df_cceb[[col for col in df_cceb.columns if col not in excluded_cols]]

errors = []

for _, row in df_cceb.iterrows():
    workorder_no = row["workorder_id"]
    filename = row["filename"]

    data = {}

    for col in df_cceb.columns:
        if col in ["workorder_id", "filename"]:
            continue

        value = row[col]
        if pd.isna(value):
            continue

        text_value = to_text(value, workorder_no, filename, col)

        data[col] = text_value

    if workorder_no not in cceb_json:
        cceb_json[workorder_no] = {
            "filename": filename,
            "data": data
        }
    else:
        cceb_json[workorder_no]["data"].update(data)

        if not cceb_json[workorder_no].get("filename"):
            cceb_json[workorder_no]["filename"] = filename


In [261]:
errors

[]

### Greasing Cardan Shaft

In [262]:
df_greasingcardanshaft = dfs.get("greasing_cardan_shaft")
greasing_cardanshaft_json = {}

excluded_cols = ["functional_location", "work_request", "target_date"]
df_greasingcardanshaft = df_greasingcardanshaft[[col for col in df_greasingcardanshaft.columns if col not in excluded_cols]]

RAW_SUFFIXES = (
    "approval_date",
    "technician_id",
    "supervisor_id",
)

def is_raw_column(col: str) -> bool:
    return col.endswith(RAW_SUFFIXES) or col.endswith("bogie_sn")


for _, row in df_greasingcardanshaft.iterrows():
    workorder_no = row["workorder_id"]
    filename = row["filename"]

    data = {}

    for col in df_greasingcardanshaft.columns:
        if col in ["workorder_id", "filename"]:
            continue

        raw_value = row[col]

        # 👇 special handling
        if is_raw_column(col):
            value = "" if pd.isna(raw_value) else str(raw_value)
        else:
            value = to_bool(raw_value)

        data[col] = value

    if workorder_no not in greasing_cardanshaft_json:
        greasing_cardanshaft_json[workorder_no] = {
            "filename": filename,
            "data": data
        }
    else:
        greasing_cardanshaft_json[workorder_no]["data"].update(data)

        if not greasing_cardanshaft_json[workorder_no].get("filename"):
            greasing_cardanshaft_json[workorder_no]["filename"] = filename


In [263]:
bool_errors

[]

### Cardan Shaft

In [264]:
df_cardanshaft = dfs.get("cardan_shaft")
cardanshaft_json = {}

excluded_cols = ["functional_location", "work_request", "target_date"]
df_cardanshaft = df_cardanshaft[[col for col in df_cardanshaft.columns if col not in excluded_cols]]

RAW_SUFFIXES = (
    "approval_date",
    "technician_id",
    "supervisor_id",
)

def is_raw_column(col: str) -> bool:
    return col.endswith(RAW_SUFFIXES) or col.endswith("bogie_sn")


for _, row in df_cardanshaft.iterrows():
    workorder_no = row["workorder_id"]
    filename = row["filename"]

    data = {}

    for col in df_cardanshaft.columns:
        if col in ["workorder_id", "filename"]:
            continue

        raw_value = row[col]

        if is_raw_column(col):
            value = "" if pd.isna(raw_value) else str(raw_value)
        else:
            value = to_bool(raw_value)

        data[col] = value

    if workorder_no not in cardanshaft_json:
        cardanshaft_json[workorder_no] = {
            "filename": filename,
            "data": data
        }
    else:
        cardanshaft_json[workorder_no]["data"].update(data)

        if not cardanshaft_json[workorder_no].get("filename"):
            cardanshaft_json[workorder_no]["filename"] = filename


In [265]:
bool_errors

[]

### Water Ponding

In [266]:
df_waterponding = dfs.get("water_ponding")
waterponding_json = {}

excluded_cols = ["functional_location", "work_request", "target_date"]
df_waterponding = df_waterponding[[col for col in df_waterponding.columns if col not in excluded_cols]]

RAW_SUFFIXES = (
    "approval_date",
    "technician_id",
    "supervisor_id",
)

def is_raw_column(col: str) -> bool:
    return col.endswith(RAW_SUFFIXES) or col.endswith("bogie_sn")

for _, row in df_waterponding.iterrows():
    workorder_no = row["workorder_id"]
    filename = row["filename"]

    data = {}

    for col in df_waterponding.columns:
        if col in ["workorder_id", "filename"]:
            continue

        raw_value = row[col]

        if is_raw_column(col):
            value = "" if pd.isna(raw_value) else str(raw_value)
        else:
            value = to_bool(raw_value)

        data[col] = value

    if workorder_no not in waterponding_json:
        waterponding_json[workorder_no] = {
            "filename": filename,
            "data": data
        }
    else:
        waterponding_json[workorder_no]["data"].update(data)

        if not waterponding_json[workorder_no].get("filename"):
            waterponding_json[workorder_no]["filename"] = filename


In [267]:
bool_errors

[]

### Train Start Up Test

In [268]:
df_trainstartuptest = dfs.get("train_startup_test")
train_startup_test = {}

excluded_cols = ["functional_location", "work_request", "target_date"]
df_trainstartuptest = df_trainstartuptest[[col for col in df_trainstartuptest.columns if col not in excluded_cols]]

RAW_SUFFIXES = (
    "4_car_train_no",
    "approval_date",
    "odometer",
    "technician_id", "technician_name",
    "supervisor_id",
)

def is_raw_column(col: str) -> bool:
    return col.endswith(RAW_SUFFIXES) or col.endswith("bogie_sn")

for _, row in df_trainstartuptest.iterrows():
    workorder_no = row["workorder_id"]
    filename = row["filename"]

    data = {}

    for col in df_trainstartuptest.columns:
        if col in ["workorder_id", "filename", "train_startup_test.stamp_id"]:
            continue

        raw_value = row[col]

        if is_raw_column(col):
            value = "" if pd.isna(raw_value) else str(raw_value)
        else:
            value = to_bool(raw_value)

        data[col] = value

    if workorder_no not in train_startup_test:
        train_startup_test[workorder_no] = {
            "filename": filename,
            "data": data
        }
    else:
        train_startup_test[workorder_no]["data"].update(data)

        if not train_startup_test[workorder_no].get("filename"):
            train_startup_test[workorder_no]["filename"] = filename


In [269]:
bool_errors

[]

### Air Standup Test

In [270]:
df_airstanduptest = dfs.get("air_standup")
airstandup_json = {}

excluded_cols = ["functional_location", "work_request", "target_date"]
df_airstanduptest = df_airstanduptest[[col for col in df_airstanduptest.columns if col not in excluded_cols]]

for _, row in df_airstanduptest.iterrows():
    workorder_no = row["workorder_id"]
    filename = row["filename"]

    data = {}
    has_data = False

    for col in df_airstanduptest.columns:
        if col in ["workorder_id", "filename"]:
            continue

        raw_value = row[col]

        if pd.isna(raw_value):
            continue

        has_data = True
        text_value = to_text(raw_value, workorder_no, filename, col)

        data[col] = text_value

    if not has_data:
        continue

    if workorder_no not in airstandup_json:
        airstandup_json[workorder_no] = {
            "filename": filename,
            "data": data
        }
    else:
        airstandup_json[workorder_no]["data"].update(data)

        if not airstandup_json[workorder_no].get("filename"):
            airstandup_json[workorder_no]["filename"] = filename


In [271]:
bool_errors

[]

### Finalize JSON

In [272]:
# tyre_pressure_json, tyre_wear_json, airbag_pressure_json, cceb_json, greasing_cardanshaft_json, cardanshaft_json, waterponding_json, train_startup_test, airstandup_json

In [273]:
import pandas as pd
import json

rows = []

json_sources = [
    ("tyre_pressure_json", tyre_pressure_json),
    ("tyre_wear_json", tyre_wear_json),
    ("airbag_pressure_json", airbag_pressure_json),
    ("cceb_json", cceb_json),
    ("greasing_cardanshaft_json", greasing_cardanshaft_json),
    ("cardanshaft_json", cardanshaft_json),
    ("waterponding_json", waterponding_json),
    ("train_startup_test", train_startup_test),
    ("airstandup_json", airstandup_json),
]

all_workorders = set()
for _, src in json_sources:
    all_workorders.update(src.keys())

for workorder_no in all_workorders:
    row = {
        "workorder_no": workorder_no,
        "filename": None,
    }

    for col_name, src in json_sources:
        payload = src.get(workorder_no, {})
        
        if row["filename"] is None:
            row["filename"] = payload.get("filename")

        row[col_name] = json.dumps(
            payload.get("data", {}),
            ensure_ascii=False
        )

    rows.append(row)

df_out = pd.DataFrame(rows)

output_file = "../../output/rsd_full_responses.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")

Saved as: ../../output/rsd_full_responses.xlsx
